In [1]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 1.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

In [3]:
def month_encoded(df: pd.DataFrame):
  df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
  df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)
  return df

In [5]:
def robust_encoded(df_train: pd.DataFrame, df_test: pd.DataFrame = None):
    cols = ['Численность населения', 'Количество домохозяйств', 'Трафик пеший, в час', 'Трафик авто, в час']
    rs = RobustScaler()
    df_train[cols] = rs.fit_transform(df_train[cols])
    if df_test is not None:
      df_test[cols] = rs.transform(df_test[cols])
    return df_train, df_test

In [6]:
def pca_applied(df_train: pd.DataFrame, df_test: pd.DataFrame = None):
  counts = [
        'Маркетплейсы, доставки, постаматы (100 м)',
        'Медицинские уч. и аптеки (300 м)',
        'Школы (300 м)',
        'Остановки (300 м)',
        'Продуктовые магазины (500 м)',
        'Пятерочки (500 м)'
    ]
  pca_param = df_train[counts]

  pca_param_train, pca_param_test = train_test_split(pca_param, test_size=0.3, random_state=598)
  pipeline = Pipeline([('scaler', StandardScaler()),('pca', PCA(n_components=4, random_state=598))])

  pipeline.fit(pca_param_train)

  pca_train = pipeline.transform(df_train.iloc[pca_param_train.index][counts])
  pca_train_valid = pipeline.transform(df_train.iloc[pca_param_test.index][counts])

  print(f'Доля объясненной дисперсии: {sum(pipeline.named_steps['pca'].explained_variance_ratio_)}')

  pca_train_df = pd.DataFrame(
      np.vstack([pca_train, pca_train_valid]),
      columns=['pca_1', 'pca_2', 'pca_3', 'pca_4'],
      index=np.concatenate([pca_param_train.index, pca_param_test.index])
  ).sort_index()

  df_train = pd.concat([df_train, pca_train_df], axis = 1)
  if df_test is not None:
    pca_test = pipeline.transform(df_test[counts])
    pca_test_df = pd.DataFrame(
        pca_test,
        columns=['pca_1', 'pca_2', 'pca_3', 'pca_4']
      )
    df_test = pd.concat([df_test, pca_test_df], axis=1)

  return df_train, df_test

In [25]:
def target_encoder_traffic_avg_bill(df_train: pd.DataFrame, df_test: pd.DataFrame = None):
    cat_cols = ['Населенный пункт', 'Регион']

    do_traffic = 'Трафик' in df_train.columns
    do_average_bill = 'Средний чек' in df_train.columns
    if not do_traffic:
        print('[target_encoder] пропуск ветки traffic: нет колонки "Трафик"')
    if not do_average_bill:
        print('[target_encoder] пропуск ветки average_bill: нет колонки "Средний чек"')

    if do_traffic:
        df_train['Населенный пункт_fold_traffic'] = np.nan
        df_train['Регион_fold_traffic'] = np.nan
    if do_average_bill:
        df_train['Населенный пункт_fold_average_bill'] = np.nan
        df_train['Регион_fold_average_bill'] = np.nan

    encoders_traffic = []
    encoders_average_bill = []

    gkf = GroupKFold(n_splits=5)

    for train_id, validation_id in gkf.split(df_train, groups=df_train['new_id']):

        X_train = df_train.iloc[train_id][['Населенный пункт', 'Регион']]
        X_validation = df_train.iloc[validation_id][['Населенный пункт', 'Регион']]

        if do_traffic:
            y_train_traffic = df_train.iloc[train_id]['Трафик']
            TE_traffic = TargetEncoder(cols=['Населенный пункт', 'Регион'], smoothing=20)
            TE_traffic.fit(X_train, y_train_traffic)
            val_traffic = TE_traffic.transform(X_validation)
            encoders_traffic.append(TE_traffic)
            df_train.iloc[validation_id, df_train.columns.get_loc('Населенный пункт_fold_traffic')] = val_traffic['Населенный пункт']
            df_train.iloc[validation_id, df_train.columns.get_loc('Регион_fold_traffic')] = val_traffic['Регион']

        if do_average_bill:
            y_train_average_bill = df_train.iloc[train_id]['Средний чек']
            TE_average_bill = TargetEncoder(cols=['Населенный пункт', 'Регион'], smoothing=20)
            TE_average_bill.fit(X_train, y_train_average_bill)
            val_average_bill = TE_average_bill.transform(X_validation)
            encoders_average_bill.append(TE_average_bill)
            df_train.iloc[validation_id, df_train.columns.get_loc('Населенный пункт_fold_average_bill')] = val_average_bill['Населенный пункт']
            df_train.iloc[validation_id, df_train.columns.get_loc('Регион_fold_average_bill')] = val_average_bill['Регион']

    if do_traffic:
        drop_cols = [c for c in ['Населенный пункт_fold_average_bill', 'Регион_fold_average_bill'] if c in df_train.columns]
        df_train_traffic = df_train.drop(columns=drop_cols)

        if df_test is not None:
            encoded_sum = None
            for enc in encoders_traffic:
                e = enc.transform(df_test[['Населенный пункт', 'Регион']])
                encoded_sum = e if encoded_sum is None else encoded_sum + e
            df_test_traffic = df_test.copy()
            df_test_traffic[['Населенный пункт_fold_traffic', 'Регион_fold_traffic']] = encoded_sum / len(encoders_traffic)
        else:
            df_test_traffic = None
    else:
        df_train_traffic = None
        df_test_traffic = None

    if do_average_bill:
        drop_cols = [c for c in ['Населенный пункт_fold_traffic', 'Регион_fold_traffic'] if c in df_train.columns]
        df_train_average_bill = df_train.drop(columns=drop_cols)

        if df_test is not None:
            encoded_sum = None
            for enc in encoders_average_bill:
                e = enc.transform(df_test[['Населенный пункт', 'Регион']])
                encoded_sum = e if encoded_sum is None else encoded_sum + e
            df_test_average_bill = df_test.copy()
            df_test_average_bill[['Населенный пункт_fold_average_bill', 'Регион_fold_average_bill']] = encoded_sum / len(encoders_average_bill)
        else:
            df_test_average_bill = None
    else:
        df_train_average_bill = None
        df_test_average_bill = None

    return df_train_traffic, df_train_average_bill, df_test_traffic, df_test_average_bill

In [26]:
def ohe_encoded(df_train: pd.DataFrame, df_test: pd.DataFrame):
    categorical_features = ['Дата открытия, категориальный', 'Торговая площадь, категориальный']

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    df_train_ohe = ohe.fit_transform(df_train[categorical_features]).astype(int)
    df_train[ohe.get_feature_names_out(categorical_features)] = df_train_ohe

    if df_test is not None:
      df_test_ohe = ohe.transform(df_test[categorical_features]).astype(int)
      df_test[ohe.get_feature_names_out(categorical_features)] = df_test_ohe

    return df_train, df_test

In [27]:
def drop(df):
  if df is None:
    return None
  cols = ['new_id', 'Месяц', 'Дата открытия, категориальный', 'Торговая площадь, категориальный', 'Населенный пункт', 'Регион']
  drop = [c for c in cols if c in df.columns]
  return df.drop(columns = drop)

In [14]:
def process_data(df_train: pd.DataFrame, df_test: pd.DataFrame = None):
  df_train = df_train.copy()
  df_test = df_test.copy() if df_test is not None else None

  df_train = month_encoded(df_train)
  if df_test is not None:
    df_test = month_encoded(df_test)

  df_train, df_test = robust_encoded(df_train, df_test)
  df_train, df_test = pca_applied(df_train, df_test)
  df_train, df_test = ohe_encoded(df_train, df_test)

  if not (('Трафик' in df_train.columns) or ('Средний чек' in df_train.columns)):
    return drop(df_train), drop(df_test)

  df_train_traffic, df_train_average_bill, df_test_traffic, df_test_average_bill = target_encoder_traffic_avg_bill(df_train, df_test)

  df_train_traffic = drop(df_train_traffic)
  df_train_average_bill = drop(df_train_average_bill)
  df_test_traffic = drop(df_test_traffic)
  df_test_average_bill = drop(df_test_average_bill)
  return df_train_traffic, df_train_average_bill, df_test_traffic, df_test_average_bill

In [29]:
df_train = pd.read_excel('Х5.xlsx')
df_test = pd.read_csv('result_test_df.csv')

In [32]:
df_train_traffic, df_train_average_bill, df_test_traffic, df_test_average_bill = process_data(df_train, df_test)

Доля объясненной дисперсии: 0.853781019527376


In [33]:
df_train_traffic.head()

,Трафик,Средний чек,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),...,pca_4,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний",Населенный пункт_fold_traffic,Регион_fold_traffic
0,59662,823.060390,-0.182274,-0.533564,-0.54306,0.321416,0,6,0,0,...,-1.215406,0,0,1,0,0,0,1,57260.693459,57482.03329
1,56674,859.361975,-0.182274,-0.533564,-0.54306,0.321416,0,6,0,0,...,-1.215406,0,0,1,0,0,0,1,57260.693459,57482.03329
2,51488,763.937766,-0.182274,-0.533564,-0.54306,0.321416,0,6,0,0,...,-1.215406,0,0,1,0,0,0,1,57260.693459,57482.03329
3,56693,836.362309,-0.182274,-0.533564,-0.54306,0.321416,0,6,0,0,...,-1.215406,0,0,1,0,0,0,1,57260.693459,57482.03329
4,58128,845.257709,-0.182274,-0.533564,-0.54306,0.321416,0,6,0,0,...,-1.215406,0,0,1,0,0,0,1,57260.693459,57482.03329


In [34]:
df_test_traffic.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),...,pca_4,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний",Населенный пункт_fold_traffic,Регион_fold_traffic
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,18.523797,0.72369,0.487842,0.056372,0,0,2,...,0.296986,1,0,0,0,1,0,0,72126.046094,72126.046094
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,18.523797,0.72369,0.487842,0.056372,0,0,2,...,0.296986,1,0,0,0,1,0,0,72126.046094,72126.046094
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,18.523797,0.72369,0.487842,0.056372,0,0,2,...,0.296986,1,0,0,0,1,0,0,72126.046094,72126.046094
3,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,18.523797,0.72369,0.487842,0.056372,0,0,2,...,0.296986,1,0,0,0,1,0,0,72126.046094,72126.046094
4,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,18.523797,0.72369,0.487842,0.056372,0,0,2,...,0.296986,1,0,0,0,1,0,0,72126.046094,72126.046094


In [19]:
df_train

,new_id,Месяц,Трафик,Средний чек,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,10,59662,823.060390,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.900,200.333333,0,6,0,0,2,0
1,0,5,56674,859.361975,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.900,200.333333,0,6,0,0,2,0
2,0,1,51488,763.937766,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.900,200.333333,0,6,0,0,2,0
3,0,6,56693,836.362309,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.900,200.333333,0,6,0,0,2,0
4,0,7,58128,845.257709,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.900,200.333333,0,6,0,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256718,21742,10,51676,1167.101083,Новый,Средний,Октябрьский рп,Волгоградская обл,6071,262,125.375,243.333333,1,1,0,0,1,0
256719,21742,11,51516,1252.914118,Новый,Средний,Октябрьский рп,Волгоградская обл,6071,262,125.375,243.333333,1,1,0,0,1,0
256720,21742,9,49593,1130.823998,Новый,Средний,Октябрьский рп,Волгоградская обл,6071,262,125.375,243.333333,1,1,0,0,1,0
256721,21742,12,52115,1461.929305,Новый,Средний,Октябрьский рп,Волгоградская обл,6071,262,125.375,243.333333,1,1,0,0,1,0
